# 05 — Differential expression

Equivalent to `scripts/05_de_analysis.py`. Produces **Figure 4**, answers **RQ2 and RQ3**.

Three levels, each answering something different:

| | Question |
|---|---|
| **A** Global, per sex | do males respond more than females? (RQ3, magnitude) |
| **B** Per cluster, per sex | which cell types respond? (RQ2) |
| **C** Overlap between sexes | do they respond with the *same* genes? (RQ3, identity) |

The project guide covers only **A**. But A alone cannot answer RQ2, and the paper's
central claim — Kenyon cells and glia respond most — lives entirely in **B**.

## A caveat to state in your report

Wilcoxon across cells treats every cell as an independent replicate. They are not:
cells from one fly brain are correlated, so p-values are anti-conservative and gene
counts inflated. Notebook 05b cross-checks this. Report direction and relative
magnitude confidently; treat absolute counts as approximate.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad

import config

sc.settings.verbosity = 3
sc.settings.figdir = config.FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)
sc.logging.print_header()

In [ ]:
adata = sc.read_h5ad(config.H5AD_ANNOTATED)
label_col = 'cell_type' if adata.obs['cell_type'].nunique() > 1 else config.LEIDEN_KEY
print(f'{adata.n_obs:,} cells, labelling by {label_col!r}')

## Helper functions

`reference='Sucrose'` makes sucrose the denominator, so **positive log2FC = up in
cocaine**. Get this backwards and every biological statement in your Discussion
inverts.

In [ ]:
def de_cocaine_vs_sucrose(subset):
    sc.tl.rank_genes_groups(subset, groupby='treatment', reference='Sucrose',
                            method=config.DE_METHOD, use_raw=True, pts=True)
    df = sc.get.rank_genes_groups_df(subset, group='Cocaine')
    return df.dropna(subset=['logfoldchanges', 'pvals_adj'])

def significant(df, lfc=config.LOG2FC_THRESHOLD):
    return df[(df['logfoldchanges'].abs() > lfc) &
              (df['pvals_adj'] < config.PADJ_THRESHOLD)]

## A. Global DE per sex

**On units:** scanpy reports log2 fold change; the paper reports natural log.
The paper's `|ln FC| > 1` equals `|log2FC| > 1.44` — *stricter* than the guide's 1.0.
We report both so the comparison is honest.

In [ ]:
results = {}

for sex in ['Male', 'Female']:
    sub = adata[adata.obs['sex'] == sex].copy()
    df = de_cocaine_vs_sucrose(sub)
    results[sex] = df
    df.to_csv(config.TABLE_DIR / f'de_global_{sex.lower()}_all.csv', index=False)

    sig = significant(df)
    sig_paper = significant(df, config.PAPER_EQUIVALENT_LOG2FC)
    print(f'{sex}: {sub.n_obs:,} cells')
    print(f'  guide threshold (|log2FC|>1.0): {len(sig)} genes '
          f'({int((sig.logfoldchanges>0).sum())} up, {int((sig.logfoldchanges<0).sum())} down)')
    print(f'  paper equivalent (|log2FC|>1.44): {len(sig_paper)} genes')
    del sub

In [ ]:
m = len(significant(results['Male']))
f = len(significant(results['Female']))
print(f'Male:Female ratio = {m}:{f}' + (f' ({m/f:.2f}x)' if f else ''))
print('Paper: 691 vs 322 (~2.1x) at their own thresholds.')
print('The RATIO is the reproducible claim; absolute counts depend on')
print('normalization, thresholds and test choice, all of which differ here.')

### Volcano plots

In [ ]:
def volcano(df, title, n_label=12):
    d = df.copy()
    d['neglog10p'] = -np.log10(d['pvals_adj'].clip(lower=1e-300))
    sig = ((d['pvals_adj'] < config.PADJ_THRESHOLD) &
           (d['logfoldchanges'].abs() > config.LOG2FC_THRESHOLD))

    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.scatter(d.loc[~sig,'logfoldchanges'], d.loc[~sig,'neglog10p'],
               s=4, c='lightgrey', rasterized=True)
    up, dn = sig & (d.logfoldchanges > 0), sig & (d.logfoldchanges < 0)
    ax.scatter(d.loc[up,'logfoldchanges'], d.loc[up,'neglog10p'], s=6,
               c='crimson', label=f'up ({int(up.sum())})', rasterized=True)
    ax.scatter(d.loc[dn,'logfoldchanges'], d.loc[dn,'neglog10p'], s=6,
               c='steelblue', label=f'down ({int(dn.sum())})', rasterized=True)
    for _, r in d[sig].nlargest(n_label, 'neglog10p').iterrows():
        ax.annotate(r['names'], (r.logfoldchanges, r.neglog10p), fontsize=6)
    ax.axhline(-np.log10(config.PADJ_THRESHOLD), ls='--', lw=.7, c='k')
    ax.axvline(config.LOG2FC_THRESHOLD, ls='--', lw=.7, c='k')
    ax.axvline(-config.LOG2FC_THRESHOLD, ls='--', lw=.7, c='k')
    ax.set_xlabel('log2FC (Cocaine / Sucrose)'); ax.set_ylabel('-log10 padj')
    ax.set_title(title, fontsize=10); ax.legend(frameon=False, fontsize=8)
    fig.tight_layout()
    return fig

for sex in ['Male', 'Female']:
    fig = volcano(results[sex], f'{sex}: cocaine vs sucrose')
    fig.savefig(config.FIG_DIR / f'05_volcano_{sex.lower()}.png', dpi=150)
    plt.show()

## B. Per-cluster DE — this is RQ2

Slow: one Wilcoxon test per cluster per sex. Clusters with fewer than
`MIN_CELLS_PER_GROUP_FOR_DE` cells in either arm are skipped, because Wilcoxon on
4 cells produces noise, not biology.

In [ ]:
per_cluster, per_cluster_genes = [], []
clusters = sorted(adata.obs[label_col].unique(), key=str)

for sex in ['Male', 'Female']:
    sex_data = adata[adata.obs['sex'] == sex]
    for cl in clusters:
        sub = sex_data[sex_data.obs[label_col] == cl].copy()
        counts = sub.obs['treatment'].value_counts()
        if min(counts.get('Cocaine', 0), counts.get('Sucrose', 0)) < config.MIN_CELLS_PER_GROUP_FOR_DE:
            per_cluster.append({'sex': sex, 'cluster': cl, 'n_cells': sub.n_obs,
                                'n_sig': np.nan, 'note': 'too few cells'})
            del sub; continue
        try:
            df = de_cocaine_vs_sucrose(sub)
        except Exception as e:
            per_cluster.append({'sex': sex, 'cluster': cl, 'n_cells': sub.n_obs,
                                'n_sig': np.nan, 'note': str(e)})
            del sub; continue
        sig = significant(df)
        per_cluster.append({'sex': sex, 'cluster': cl, 'n_cells': sub.n_obs,
                            'n_sig': len(sig),
                            'n_up': int((sig.logfoldchanges > 0).sum()),
                            'n_down': int((sig.logfoldchanges < 0).sum()), 'note': ''})
        if len(sig):
            s = sig.copy(); s['sex'], s['cluster'] = sex, cl
            per_cluster_genes.append(s)
        del sub
    print(f'{sex}: done')

pc = pd.DataFrame(per_cluster)
pc.to_csv(config.TABLE_DIR / 'de_per_cluster_summary.csv', index=False)
if per_cluster_genes:
    pd.concat(per_cluster_genes, ignore_index=True).to_csv(
        config.TABLE_DIR / 'de_per_cluster_significant_genes.csv', index=False)

In [ ]:
ranked = (pc.dropna(subset=['n_sig']).groupby('cluster', observed=True)['n_sig']
            .sum().sort_values(ascending=False))
print('Most cocaine-responsive clusters:')
print(ranked.head(10).to_string())
print()
print('Paper: Kenyon cells (C11), astrocytes (C17), surface glia (C22),')
print('and unannotated C16 were their strongest responders.')
print('Match by CELL TYPE, not cluster number.')

In [ ]:
pivot = pc.pivot_table(index='cluster', columns='sex', values='n_sig').fillna(0)
fig, ax = plt.subplots(figsize=(5, max(4, 0.25*len(pivot))))
im = ax.imshow(pivot.values, aspect='auto', cmap='magma')
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=6)
ax.set_title('Significant DE genes per cluster', fontsize=10)
fig.colorbar(im, ax=ax, label='n DE genes'); fig.tight_layout()
fig.savefig(config.FIG_DIR / '05_de_burden_heatmap.png', dpi=150)

## C. Sexual dimorphism — shared vs distinct

The paper's subtler point: some shared genes move in **opposite directions** between
sexes. That is stronger evidence of dimorphism than counts alone.

In [ ]:
male_sig, female_sig = significant(results['Male']), significant(results['Female'])
ms, fs = set(male_sig['names']), set(female_sig['names'])
shared = ms & fs

print(f'male only:   {len(ms - fs)}')
print(f'female only: {len(fs - ms)}')
print(f'shared:      {len(shared)}')

In [ ]:
if shared:
    cmp = pd.DataFrame({
        'male_log2FC': male_sig.set_index('names')['logfoldchanges'][list(shared)],
        'female_log2FC': female_sig.set_index('names')['logfoldchanges'][list(shared)],
    })
    cmp['opposite'] = np.sign(cmp.male_log2FC) != np.sign(cmp.female_log2FC)
    cmp.to_csv(config.TABLE_DIR / 'de_shared_genes_direction.csv')
    print(f'{int(cmp.opposite.sum())} shared genes change in OPPOSITE directions')

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(cmp.male_log2FC, cmp.female_log2FC, s=14,
               c=np.where(cmp.opposite, 'crimson', 'grey'))
    lim = float(np.nanmax(np.abs(cmp[['male_log2FC','female_log2FC']].values))) * 1.1
    ax.plot([-lim, lim], [-lim, lim], ls='--', lw=.7, c='k')
    ax.axhline(0, lw=.5, c='k'); ax.axvline(0, lw=.5, c='k')
    ax.set_xlabel('male log2FC'); ax.set_ylabel('female log2FC')
    fig.tight_layout()
    fig.savefig(config.FIG_DIR / '05_sex_concordance.png', dpi=150)
    display(cmp.sort_values('opposite', ascending=False).head(15).round(3))

## D. Positive control

Genes the paper names explicitly. If **none** of these move at all, suspect a
pipeline error before you conclude the paper was wrong.

In [ ]:
watch = (config.PAPER_CORE_RESPONSE_GENES['up'] +
         config.PAPER_CORE_RESPONSE_GENES['down'] +
         config.PAPER_DISCUSSION_GENES)

rows = []
for sex in ['Male', 'Female']:
    d = results[sex].set_index('names')
    for g in watch:
        if g in d.index:
            r = d.loc[g]
            rows.append({'gene': g, 'sex': sex,
                         'log2FC': round(float(r.logfoldchanges), 3),
                         'padj': float(r.pvals_adj)})

chk = pd.DataFrame(rows)
chk.to_csv(config.TABLE_DIR / 'paper_gene_check.csv', index=False)
chk.pivot_table(index='gene', columns='sex', values='log2FC').round(2)